<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Midbody Speed Against the Dataset's Own

The check written down in the journal of 2026-10-14, *A Midbody Speed*, before it ran: on every block-1 file fetched
and MD5-checked at the time of the run except 1024910, on which the span was chosen, the lab's eight-arc midbody speed
(the Rust binary `midbody-speed`) passes if in every record its Pearson correlation with the published `midbody_speed`
is at least 0.95 and the slope of a least-squares line through the origin, the lab's on the published, lies between
0.9 and 1.1, over the frames where both exist. Sign agreement and the median absolute difference are reported, not
tested.

Run from the repository's root with the block-1 files in `data/raw/owmd-n2/` (fetched by `fetch-block`, which names
a file only once its MD5 matches) and the Rust toolchain installed; the notebook builds and calls the binary.

In [1]:
import os, subprocess
import numpy as np
import h5py

os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DIRECTORY, SUFFIX, EXPLORED = "data/raw/owmd-n2", "_features.hdf5", "1024910"
# The seventeen records fetched and MD5-checked when the check ran (journal of 2026-10-14, *A Midbody Speed*),
# pinned so that a re-run checks the same set however many files have arrived since.
records = ["1006562", "1007123", "1007405", "1008698", "1011246", "1011901", "1013148", "1014831", "1020507",
           "1021835", "1024910", "1025254", "1025397", "1029332", "1030984", "1031252", "1032937"]
missing = [record for record in records if not os.path.exists(f"{DIRECTORY}/{record}{SUFFIX}")]
assert not missing, f"fetch these first: {missing}"
held_out = [record for record in records if record != EXPLORED]
subprocess.run(["cargo", "build", "--quiet", "--release", "--bin", "midbody-speed"], check=True)
print(len(records), "records checked;", len(held_out), "held out:", " ".join(held_out))
print("h5py", h5py.__version__, "numpy", np.__version__)

17 records checked; 16 held out: 1006562 1007123 1007405 1008698 1011246 1011901 1013148 1014831 1020507 1021835 1025254 1025397 1029332 1030984 1031252 1032937
h5py 3.16.0 numpy 2.5.4


In [2]:
def lab_speeds(record):
    out = subprocess.run(["target/release/midbody-speed", record], check=True, capture_output=True, text=True).stdout
    return np.array([float(line.split("\t")[1]) for line in out.splitlines()[1:]])

def published_speeds(record):
    with h5py.File(f"{DIRECTORY}/{record}{SUFFIX}", "r") as file:
        return file["/features_timeseries"]["midbody_speed"].astype(float)

rows = []
for record in held_out:
    ours, theirs = lab_speeds(record), published_speeds(record)
    assert len(ours) == len(theirs), (record, len(ours), len(theirs))
    both = ~np.isnan(ours) & ~np.isnan(theirs)
    a, b = ours[both], theirs[both]
    r = np.corrcoef(a, b)[0, 1]
    slope = np.sum(a * b) / np.sum(b * b)
    rows.append((record, both.sum(), r, slope, np.mean(np.sign(a) == np.sign(b)), np.median(np.abs(a - b)),
                 r >= 0.95 and 0.9 <= slope <= 1.1))
print("record   frames  r      slope  sign   median|diff| um/s  passes")
for record, n, r, slope, sign, diff, ok in rows:
    print(f"{record}  {n:6d}  {r:.3f}  {slope:.3f}  {sign:.3f}  {diff:6.2f}             {ok}")
print("every record passes:", all(row[-1] for row in rows))
print("medians over records: r %.3f, slope %.3f, sign %.3f, |diff| %.2f" % tuple(
    np.median([row[i] for row in rows]) for i in (2, 3, 4, 5)))

record   frames  r      slope  sign   median|diff| um/s  passes
1006562   25061  0.979  0.981  0.911   11.97             True
1007123   23970  0.986  1.002  0.879   11.10             True
1007405   26337  0.947  0.930  0.798   10.06             False
1008698   13093  0.990  1.002  0.934   12.10             True
1011246  166840  0.961  0.954  0.813   10.52             True
1011901   24452  0.969  0.972  0.873   11.25             True
1013148   19855  0.982  1.001  0.923   12.70             True
1014831   24097  0.976  0.990  0.836   10.55             True
1020507   22126  0.992  1.012  0.905   10.38             True
1021835   16712  0.989  1.010  0.953   12.61             True
1025254    9652  0.994  1.016  0.960   14.80             True
1025397   14076  0.960  0.950  0.839   10.86             True
1029332   13002  0.970  0.962  0.838   12.29             True
1030984   13190  0.958  0.938  0.810   12.64             True
1031252   10362  0.991  1.012  0.933   14.86             True
10329